In [ ]:
import tensorflow as tf
import pathlib
import matplotlib.pyplot as plt
import os
import numpy as np
from dotenv import load_dotenv

"""
1) Buscar as máscaras - OK
2) Carregar e pré-processar as máscaras - OK
3) Criar pipeline para Aumento de Dados - OK
4) Configurar um dataset baseado nas máscaras - OK
5) Formular um treinamento - PENDENTE
6) Visualizar imagens - OK
"""

### 1. Configurar o caminho e criar lista de imagens ###
load_dotenv()

img_dir = os.getenv("BASE_IMG_FOLDER")
img_dir = pathlib.Path(img_dir)
mask_dir = os.getenv("OUTPUT_FOLDER")
mask_dir = pathlib.Path(mask_dir)

# Verificar se o diretório existe
if not os.path.exists(mask_dir):
    raise ValueError(f"Diretório não encontrado: {mask_dir}")

# Função para extrair o número do nome do arquivo
def get_file_number(filepath):
    return int(os.path.basename(filepath).replace('.png', ''))
def get_file_number_mask(filepath):
    return int(os.path.basename(filepath).replace('.npy', ''))

# Verificar se os arquivos existem e converter PosixPath para strings 
mask_files = []
for path in mask_dir.glob('*.npy'):
    if os.path.exists(path):
        mask_files.append(str(path))

image_files = []
for path in img_dir.glob('*.png'):
    if os.path.exists(path):
        image_files.append(str(path))

# Ordenar a lista usando o número do arquivo como chave
image_files = sorted(image_files, key=get_file_number)
mask_files = sorted(mask_files, key=get_file_number_mask)

if len(image_files) != len(mask_files):
    raise ValueError("El número de imágenes y máscaras no coincide")

mask_count = len(mask_files)
print(f"Encontradas {mask_count} máscaras")

# Verificar se temos imagens
if mask_count == 0:
    raise ValueError(f"Nenhuma imagem PNG encontrada em: {mask_dir}")

# Mostrar alguns caminhos de exemplo
print("\nPrimeiros 5 caminhos de máscaras:")
for path in mask_files[:5]:
    print(path)

In [2]:
import numpy as np
# Linux
mask = np.load(r'/media/renato/Data/PIBIC/Results/000.npy')

# Windows
# mask = np.load('C:\\Users\\Renato\\Documents\\PIBIC\\Results\\000.npy')

In [ ]:
mask.shape

In [ ]:
plt.imshow(mask[:,:,0])

In [ ]:
def load_and_preprocess_image_mask(image_path, mask_path):
    try:
        # Carregar imagem RGB
        img = tf.io.read_file(image_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, [256, 256])  # Redimensionar o tamanho
        img = tf.cast(img, tf.float32) / 255.0  # Normalizar a imagem

        # Carregar máscara do arquivo numpy
        mask = np.load(mask_path.numpy().decode())  # Ler a máscara
        mask = tf.convert_to_tensor(mask, dtype=tf.float32)  # Converter como tensor

        # Normalizar máscara
        mask = mask / 255.0  # Converter valores de 0-255 a 0-1
        mask = tf.where(mask >= 0.3, 1.0, 0.0)  # Binarizar a máscara

        return img, mask
    except Exception as e:
        print(f"Error al procesar imagen/máscara {image_path}, {mask_path}: {str(e)}")
        raise


## Função para aplicar aumento de dados considerando máscaras como arrays multicamadas
def augment_image_mask(image, mask):
    # Gere um valor aleatório comum para transformações
    flip_left_right = tf.random.uniform(shape=[]) > 0.5
    flip_up_down = tf.random.uniform(shape=[]) > 0.5
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)  # 0, 90, 180, 270 graus

    # Flip horizontal
    if flip_left_right:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # Flip vertical
    if flip_up_down:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    # Aplicar rotação à imagem
    image = tf.image.rot90(image, k=k)

    # Aplique rotação à máscara em toda a sua dimensão
    mask = tf.image.rot90(mask, k=k)  # Agora giramos a máscara inteira de uma vez

    # Configurações adicionais de imagem
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, 0.8, 1.2)

    return image, mask

# Converter listas de rotas em conjunto de dados do TensorFlow
def process_path(image_path, mask_path):
    img, mask = tf.py_function(load_and_preprocess_image_mask, [image_path, mask_path], [tf.float32, tf.float32])
    img.set_shape((256, 256, 3))  # Defina o tamanho esperado da imagem
    mask.set_shape((256, 256, 18))  # Defina o tamanho esperado da máscara
    return img, mask

# Criar um conjunto de dados a partir de listas de imagens e máscaras
train_dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
train_dataset = train_dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

# Aplicar Aumento de Dados
train_dataset = train_dataset.map(lambda img, mask: augment_image_mask(img, mask), num_parallel_calls=tf.data.AUTOTUNE)

# Configurar o conjunto de dados
BUFFER_SIZE = 100   # Talvez aumentar para 128 no futuro
BATCH_SIZE = 32     # Talvez aumentar para 64 no futuro

train_dataset = (
    train_dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Inspecionar o conjunto de dados
for img, mask in train_dataset.take(1):
    print(f"Imagen batch: {img.shape}")  # Esperado: (batch_size, 256, 256, 3)
    print(f"Máscara batch: {mask.shape}")  # Esperado: (batch_size, 256, 256, 18)

In [ ]:
mask[0]

In [ ]:
def unet_multilabel_improved(input_shape=(256, 256, 3), num_classes=18):
    inputs = tf.keras.layers.Input(shape=input_shape)

    # Encoder (Downsampling)
    def conv_block(x, filters, dropout_rate=0.1):
        x = tf.keras.layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
        return x

    c1 = conv_block(inputs, 64)
    p1 = tf.keras.layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 128)
    p2 = tf.keras.layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 256)
    p3 = tf.keras.layers.MaxPooling2D((2, 2))(c3)

    # Bottleneck
    c4 = conv_block(p3, 512, dropout_rate=0.3)

    # Decoder (Upsampling)
    def upsample_block(x, skip_connection, filters):
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        x = tf.keras.layers.Conv2D(filters, (2, 2), activation='relu', padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Concatenate()([x, skip_connection])
        x = conv_block(x, filters)
        return x

    u5 = upsample_block(c4, c3, 256)
    u6 = upsample_block(u5, c2, 128)
    u7 = upsample_block(u6, c1, 64)

    # Salida con activación sigmoide para máscaras binarias multilabel
    outputs = tf.keras.layers.Conv2D(num_classes, (1, 1), activation='sigmoid')(u7)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

# Crear el modelo
model = unet_multilabel_improved()
model.summary()

In [8]:
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_pred = tf.keras.backend.clip(y_pred, 1e-7, 1 - 1e-7)
        focal_loss = -alpha * y_true * tf.keras.backend.pow(1 - y_pred, gamma) * tf.keras.backend.log(y_pred)
        focal_loss -= (1 - alpha) * (1 - y_true) * tf.keras.backend.pow(y_pred, gamma) * tf.keras.backend.log(1 - y_pred)
        return tf.keras.backend.mean(focal_loss)
    return loss

In [ ]:
for img, mask in train_dataset.take(1):
    print(f"Forma de la máscara: {mask.shape}")  # Deve ser (256, 256, 18)
    print(f"Tipo de datos de la máscara: {mask.dtype}")  # Deve ser float32
    print(f"Valores únicos en la máscara: {tf.unique(tf.reshape(mask, [-1]))[0].numpy()}")  # Deve conter valores 0 a 1

In [ ]:
## Obter o número total de amostras no conjunto de dados sem lote prévio
dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
dataset_size = tf.data.experimental.cardinality(dataset).numpy()

# Definir proporções para treinar, validar e testar
train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size

# Mesclar o conjunto de dados antes de dividir
shuffled_dataset = dataset.shuffle(BUFFER_SIZE)
shuffled_dataset = shuffled_dataset.shuffle(BUFFER_SIZE)

# Divida o conjunto de dados ANTES de aplicar o lote para evitar o problema de lote duplo
train_dataset = shuffled_dataset.take(train_size)
remaining_dataset = shuffled_dataset.skip(train_size)
val_dataset = remaining_dataset.take(val_size)
test_dataset = remaining_dataset.skip(val_size)

# Aplicar aumento de dados apenas ao conjunto de treinamento
train_dataset = train_dataset.map(lambda img, mask: augment_image_mask(img, mask), num_parallel_calls=tf.data.AUTOTUNE)

# Aplicar lote após divisão
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Confirmar os tamanhos do conjunto
print(f'Tamaño del dataset de entrenamiento: {train_size}')
print(f'Tamaño del dataset de validación: {val_size}')
print(f'Tamaño del dataset de evaluación: {test_size}')


In [ ]:
for img, mask in train_dataset.take(1):
    print(f"Forma de la imagen (batch): {img.shape}")  # Deve ser (BATCH_SIZE, 256, 256, 3)
    print(f"Forma de la máscara (batch): {mask.shape}")  # Deve ser (BATCH_SIZE, 256, 256, 18)

In [ ]:
# Callbacks para mejorar el entrenamiento
callbacks = [
    # Early stopping para prevenir el sobreajuste
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,  # Permitir más épocas antes de detenerse si no hay mejora
        restore_best_weights=True,
        verbose=1
    ),

    # Guardar el mejor modelo basado en la menor pérdida de validación
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model_focal.keras',
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=1
    ),

    # Reducir la tasa de aprendizaje si la pérdida de validación se estanca
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,  # Reducir la tasa de aprendizaje a la mitad si no hay mejora
        patience=5,  # Esperar 5 épocas sin mejora antes de reducir
        min_lr=1e-6,  # Límite inferior para la tasa de aprendizaje
        verbose=1
    ),

    # Registrar métricas en TensorBoard
    tf.keras.callbacks.TensorBoard(
        log_dir='logs_focal',
        histogram_freq=1,
        write_graph=True,
        write_images=True
    )
]

# Tasa de aprendizaje inicial y programador de decaimiento exponencial
# Causa do erro: 
# Definição da taxa de aprendizagem diretamente, mas no otimizador foi criado
# com um LearningRateSchedule.
initial_learning_rate = 1e-3
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=1000,
    decay_rate=0.9,
    staircase=True
)

# Compilação antes do erro
# # Compilar el modelo con focal loss y métricas específicas
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
#     loss=focal_loss(),
#     metrics=['accuracy', 
#              tf.keras.metrics.MeanIoU(num_classes=18),
#              tf.keras.metrics.Precision(), 
#              tf.keras.metrics.Recall()]
# )

# Sugestão de solução
# Recompilar o modelo com novo otimizador para ajuste fino
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # Nova taxa de aprendizagem
    loss=focal_loss(),
    metrics=[
        'accuracy', 
        tf.keras.metrics.MeanIoU(num_classes=18),
        tf.keras.metrics.Precision(), 
        tf.keras.metrics.Recall()
        ]
)

# Definir la estrategia de entrenamiento en fases (fine-tuning)
EPOCHS_PHASE_1 = 75  # Entrenamiento inicial
EPOCHS_PHASE_2 = 25  # Ajuste fino

print("Entrenamiento: Fase 1 - Aprendizaje inicial")
history_1 = model.fit(
    train_dataset,
    epochs=EPOCHS_PHASE_1,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

# Reducir la tasa de aprendizaje para la fase de ajuste fino
tf.keras.backend.set_value(model.optimizer.learning_rate, 1e-4) # Otimizador setado

print("Entrenamiento: Fase 2 - Fine-tuning con tasa de aprendizaje reducida")
history_2 = model.fit(
    train_dataset,
    epochs=EPOCHS_PHASE_2,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

# Visualización de la evolución del entrenamiento
import matplotlib.pyplot as plt

def plot_training_metrics(history_1, history_2):
    combined_history = {
        'loss': history_1.history['loss'] + history_2.history['loss'],
        'val_loss': history_1.history['val_loss'] + history_2.history['val_loss'],
        'accuracy': history_1.history['accuracy'] + history_2.history['accuracy'],
        'val_accuracy': history_1.history['val_accuracy'] + history_2.history['val_accuracy'],
        'precision': history_1.history['precision'] + history_2.history['precision'],
        'val_precision': history_1.history['val_precision'] + history_2.history['val_precision'],
        'recall': history_1.history['recall'] + history_2.history['recall'],
        'val_recall': history_1.history['val_recall'] + history_2.history['val_recall'],
    }

    plt.figure(figsize=(15, 5))
    
    # Pérdida
    plt.subplot(1, 3, 1)
    plt.plot(combined_history['loss'], label='Pérdida de entrenamiento')
    plt.plot(combined_history['val_loss'], label='Pérdida de validación')
    plt.title('Evolución de la pérdida')
    plt.legend()

    # Precisión
    plt.subplot(1, 3, 2)
    plt.plot(combined_history['accuracy'], label='Precisión de entrenamiento')
    plt.plot(combined_history['val_accuracy'], label='Precisión de validación')
    plt.title('Evolución de la precisión')
    plt.legend()

    # Recall
    plt.subplot(1, 3, 3)
    plt.plot(combined_history['recall'], label='Recall de entrenamiento')
    plt.plot(combined_history['val_recall'], label='Recall de validación')
    plt.title('Evolución del Recall')
    plt.legend()

    plt.show()

plot_training_metrics(history_1, history_2)



In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Perda no treinamento')
plt.plot(history.history['val_loss'], label='Perda na validação')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.legend()
plt.show()


In [ ]:
import datetime
# Obtener la fecha y hora actual en formato YYYY-MM-DD_HH-MM-SS
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Crear el nombre del archivo con la fecha y hora actuales
model_filename = f'modelo_segmentacion_{current_time}.h5'

# Guardar el modelo
model.save(model_filename)

print(f"Modelo guardado como: {model_filename}")

In [ ]:
import json

history_filename = f'history_{current_time}.json'

# Guardar el historial de entrenamiento como un archivo JSON
with open(history_filename, 'w') as f:
    json.dump(history.history, f)

print(f"Modelo guardado como: {model_filename}")
print(f"Historial guardado como: {history_filename}")

In [ ]:
loss, accuracy, iou = model.evaluate(test_dataset)
print(f'Pérdida en evaluación: {loss}')
print(f'Precisión en evaluación: {accuracy}')
print(f'IoU en evaluación: {iou}')

# 1st RETRAIN

In [ ]:
# Carregar o modelo previamente treinado
model = tf.keras.models.load_model(
    'modelo_segmentacion_2025-01-24_07-13-34.h5', 
    custom_objects={'MeanIoU': tf.keras.metrics.MeanIoU}
)

print("Modelo cargado exitosamente.")
model.summary()

In [ ]:
# Configurar un nuevo optimizador con una tasa de aprendizaje reducida
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',  # Pérdida adecuada para multilabel
              metrics=['accuracy', tf.keras.metrics.MeanIoU(num_classes=18)])

# Definir el número adicional de épocas
EPOCHS_EXTRA = 25

history = model.fit(
    train_dataset,
    epochs=EPOCHS_EXTRA,
    validation_data=val_dataset,
    verbose=1
)

import datetime
# Obtener la fecha y hora actual en formato YYYY-MM-DD_HH-MM-SS
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Crear el nombre del archivo con la fecha y hora actuales
model_filename = f'modelo_segmentacion_retrained_{current_time}.h5'

# Guardar el modelo
model.save(model_filename)

print(f"Modelo guardado como: {model_filename}")

import json

history_filename = f'history_retrained_{current_time}.json'

# Guardar el historial de entrenamiento como un archivo JSON
with open(history_filename, 'w') as f:
    json.dump(history.history, f)


print(f"Historial guardado como: {history_filename}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Perda no treinamento')
plt.plot(history.history['val_loss'], label='Perda na validação')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def visualize_multilabel_predictions(model, dataset):
    for img, mask in dataset.take(1):
        pred_mask = model.predict(tf.expand_dims(img[0], axis=0))[0]

        plt.figure(figsize=(10, 5))

        # Imagen original
        plt.subplot(1, 3, 1)
        plt.imshow(img[0].numpy())
        plt.title("Imagen Original")

        # Visualizar clase 0 real y predicha
        plt.subplot(1, 3, 2)
        plt.imshow(mask[0][:, :, 16], cmap='gray')
        plt.title("Máscara Real (Clase 0)")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask[:, :, 16], cmap='gray')
        plt.title("Máscara Predicha (Clase 0)")

        plt.show()
        break

visualize_multilabel_predictions(model, train_dataset)


# 2nd RETRAIN

In [ ]:
# Carregar o modelo previamente treinado
model = tf.keras.models.load_model(
    'modelo_segmentacion_retrained_2025-01-27_06-39-03.h5', 
    custom_objects={'MeanIoU': tf.keras.metrics.MeanIoU}
)

print("Modelo cargado exitosamente.")
model.summary()

In [ ]:
# Configurar un nuevo optimizador con una tasa de aprendizaje reducida
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',  # Pérdida adecuada para multilabel
              metrics=['accuracy', tf.keras.metrics.MeanIoU(num_classes=18)])

# Definir el número adicional de épocas
EPOCHS_EXTRA = 50

history = model.fit(
    train_dataset,
    epochs=EPOCHS_EXTRA,
    validation_data=val_dataset,
    verbose=1
)

import datetime
# Obtener la fecha y hora actual en formato YYYY-MM-DD_HH-MM-SS
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Crear el nombre del archivo con la fecha y hora actuales
model_filename = f'modelo_segmentacion_retrained_{current_time}.h5'

# Guardar el modelo
model.save(model_filename)

print(f"Modelo guardado como: {model_filename}")

import json

history_filename = f'history_retrained_{current_time}.json'

# Guardar el historial de entrenamiento como un archivo JSON
with open(history_filename, 'w') as f:
    json.dump(history.history, f)


print(f"Historial guardado como: {history_filename}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Perda no treinamento')
plt.plot(history.history['val_loss'], label='Perda na validação')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def visualize_multilabel_predictions(model, dataset):
    for img, mask in dataset.take(1):
        pred_mask = model.predict(tf.expand_dims(img[0], axis=0))[0]

        plt.figure(figsize=(10, 5))

        # Imagen original
        plt.subplot(1, 3, 1)
        plt.imshow(img[0].numpy())
        plt.title("Imagen Original")

        # Visualizar clase 0 real y predicha
        plt.subplot(1, 3, 2)
        plt.imshow(mask[0][:, :, 17], cmap='gray')
        plt.title("Máscara Real (Clase 0)")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask[:, :, 17], cmap='gray')
        plt.title("Máscara Predicha (Clase 0)")

        plt.show()
        break

visualize_multilabel_predictions(model, train_dataset)
